In [1]:
import yfinance as yf
import pandas as pd
import requests

In [ ]:
headers = {
"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
"Accept": "application/json, text/plain, */*",
"Origin": "https://www.nasdaq.com",
"Referer": "https://www.nasdaq.com/",
}
res = requests.get('https://api.nasdaq.com/api/screener/stocks?tableonly=true&limit=8000&offset=0', headers=headers)


In [ ]:
data = res.json()
df = pd.DataFrame(data=data['data']['table']['rows'])
df = df.query("marketCap != 'NA'")
len(df)
top500 = df.sort_values(by='marketCap', ascending=False)[:5000]
top500.head()
d1 = top500.query("symbol in ('GOOGL', 'RKLB')")

d1.columns

In [ ]:
historic_data = yf.Ticker("AAPL")
historic_data.fast_info['dayHigh']
# print(historic_data.head())

In [ ]:
hist = yf.download('AAPL', period = '10y', progress=False)
hist.columns

In [ ]:
df = yf.download(
    "AAPL",
    period="max",
    auto_adjust=True,
    progress=False
)
df.head()

In [ ]:
df.columns = df.columns.droplevel(1)

In [ ]:
print(df.columns)
small_df = df.head(2)
small_df = small_df.reset_index()
print(small_df.columns)

In [ ]:
small_df.to_json(orient="records", date_format="iso")

In [ ]:
import json
json.loads(small_df.to_json(orient="records", date_format="iso", date_unit="s"))

In [ ]:
import pandas as pd
import yfinance as yf

# 1. Define list of stock tickers
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN"]

all_companies_data = []

for symbol in tickers:
    try:
        print(f"Fetching data for {symbol}...")
        stock = yf.Ticker(symbol)
        
        # Extract financial statements
        inc = stock.quarterly_income_stmt
        bs = stock.quarterly_balance_sheet
        cf = stock.quarterly_cashflow
        
        # Helper to safely extract line items across names
        def get_row(df, names):
            for name in names:
                if name in df.index:
                    return df.loc[name]
            return pd.Series(dtype=float)

        # Retrieve core metrics
        revenue = get_row(inc, ['Total Revenue', 'Operating Revenue'])
        net_income = get_row(inc, ['Net Income Common Stockholders', 'Net Income'])
        operating_income = get_row(inc, ['Operating Income'])
        gross_profit = get_row(inc, ['Gross Profit'])
        
        total_assets = get_row(bs, ['Total Assets'])
        total_liabilities = get_row(bs, ['Total Liabilities Net Minority Interest', 'Total Liabilities'])
        stockholder_equity = get_row(bs, ['Stockholders Equity', 'Total Equity Gross Minority Interest'])
        total_debt = get_row(bs, ['Total Debt'])
        cash_and_equiv = get_row(bs, ['Cash Cash Equivalents And Short Term Investments', 'Cash And Cash Equivalents'])
        
        operating_cash_flow = get_row(cf, ['Operating Cash Flow', 'Total Cash From Operating Activities'])
        capital_expenditures = get_row(cf, ['Capital Expenditure', 'Capital Expenditures'])
        free_cash_flow = get_row(cf, ['Free Cash Flow'])

        # Ratios & Margins
        profit_margin = (net_income / revenue) * 100
        operating_margin = (operating_income / revenue) * 100
        debt_to_equity = total_debt / stockholder_equity
        roe = (net_income / stockholder_equity) * 100

        # Combine all metrics into a single dictionary per ticker
        metrics_dict = {
            'Total Revenue': revenue,
            'Net Income': net_income,
            'Net Profit Margin (%)': profit_margin,
            'Operating Margin (%)': operating_margin,
            'Total Assets': total_assets,
            'Total Liabilities': total_liabilities,
            'Stockholders Equity': stockholder_equity,
            'Total Debt': total_debt,
            'Cash & Short Term Investments': cash_and_equiv,
            'Debt to Equity Ratio': debt_to_equity,
            'Return on Equity - ROE (%)': roe,
            'Operating Cash Flow': operating_cash_flow,
            'Capital Expenditures': capital_expenditures,
            'Free Cash Flow': free_cash_flow,
        }

        # Format rows into column headers: metric-name-YYYY-MM-DD
        company_row = {'SYMBOL': symbol}
        
        for metric_name, series in metrics_dict.items():
            if not series.empty:
                for date, value in series.items():
                    # Format date to YYYY-MM-DD string
                    date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
                    col_name = f"{metric_name.lower().replace(' ', '-')}-{date_str}"
                    company_row[col_name] = value

        all_companies_data.append(company_row)

    except Exception as e:
        print(f"Error processing {symbol}: {e}")

# 2. Build final DataFrame and export to CSV
final_df = pd.DataFrame(all_companies_data)

# Export to CSV file
csv_filename = "value_investing_quarterly_data.csv"
final_df.to_csv(csv_filename, index=False)
print(f"\nSuccessfully generated and exported data to {csv_filename}!")

In [ ]:
!pip install nsepython

In [26]:
import nsepython
data = nsepython.nse_eq("TCS")
data

{}